# 04 - Produto: dados da página de projeções

Gera `reports/pagina/produto_dados.js`, lido por `reports/pagina/index.html` (projeções). Rodar todo mês, depois do `03b` (que
atualiza `backtest_ecmwf.pkl`) e com o deck do NEWAVE e a rodada do ECMWF do mês.

- Projeção: deck mais recente com rodada do ECMWF do mesmo mês (D031), M+1 a M+6; esperado (média `precip`), LI e LS
  (P10 / P90 mensal da ENA dos membros recentrados); probabilidade com σ por grupo de horizonte.
- Patamares de um ano ainda não publicado: mesmo mês do ano anterior (D032).
- Histórico oficial: `data/raw/bandeiras/historico.csv` (ANEEL), mais a bandeira do mês corrente pelo boletim da CCEE.
- Desempenho: backtest sem vazamento da variante A (`03b`).

In [1]:
import json
import warnings
from datetime import date

import numpy as np
import pandas as pd

from forecast_bandeiras import backtest, ecmwf, load, newave, regra

warnings.filterwarnings("ignore", message="patamares de")
ORD = ["Verde", "Amarela", "Vermelha 1", "Vermelha 2"]
base = pd.read_parquet(load.DATA_PROCESSED / "base_mensal.parquet")
base.index = base.index.to_period("M")
base["cor_oficial"] = [c if isinstance(c, str) else r for c, r in zip(base.cor_boletim, base.cor_regra)]
A = pd.read_pickle(load.DATA_PROCESSED / "backtest_ecmwf.pkl")["A: ECMWF média"].reset_index(drop=True)

rodadas = set(ecmwf.rodadas_disponiveis())
dk = max(pd.Period(f"{a}-{m:02d}", "M") for a, m in backtest.decks_disponiveis() if pd.Period(f"{a}-{m:02d}", "M") in rodadas)
base = base.reindex(base.index.union(pd.period_range(dk + 1, dk + 6, freq="M")))
pr = backtest.preparar_deck(dk.year, dk.month, base)
refs = {"esp": ecmwf.referencia_media(dk, base), "li": ecmwf.referencia_quantil(dk, base, 0.1),
        "ls": ecmwf.referencia_quantil(dk, base, 0.9)}
cen = backtest.prever_cenarios(pr, refs)
cen = cen[(cen.h >= 1) & (cen.h <= 6)].reset_index(drop=True)
dk, cen[["alvo", "cor_esp", "cor_li", "cor_ls"]]

(Period('2026-09', 'M'),
       alvo cor_esp   cor_li cor_ls
 0  2026-10   Verde  Amarela  Verde
 1  2026-11   Verde    Verde  Verde
 2  2026-12   Verde    Verde  Verde
 3  2027-01   Verde    Verde  Verde
 4  2027-02   Verde    Verde  Verde
 5  2027-03   Verde    Verde  Verde)

## Histórico oficial e valores do adicional

In [2]:
hist = pd.read_csv(load.DATA_RAW / "bandeiras" / "historico.csv", sep=";", decimal=",")
NOMES = {"Verde": "Verde", "Amarela": "Amarela", "Vermelha P1": "Vermelha 1", "Vermelha P2": "Vermelha 2"}
def nome(n):
    try:
        n = n.encode("latin-1").decode("utf-8")  # o arquivo vem com acentos em dupla codificação
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    return NOMES.get(n, "Escassez hídrica" if n.startswith("Escassez") else n)
hist = pd.DataFrame({"mes": pd.PeriodIndex(hist.DatCompetencia.str[:7], freq="M"),
                     "cor": hist.NomBandeiraAcionada.map(nome), "adicional": hist.VlrAdicionalBandeira}).set_index("mes")
# conferência com a regra reconstruída (2023 em diante)
comum = hist.index.intersection(base.index[base.cor_oficial.notna()])
print("divergências com a base:", int((hist.loc[comum, "cor"] != base.loc[comum, "cor_oficial"]).sum()), "de", len(comum))
# valores vigentes do adicional (R$/MWh): o último valor de cada cor
vigente = {c: float(hist[hist.cor == c].adicional.iloc[-1]) for c in ORD}
vigente["Verde"] = 0.0
# meses já anunciados que ainda não estão no arquivo da ANEEL: bandeira do boletim da CCEE
for m in base.index[(base.index > hist.index.max()) & base.cor_oficial.notna()]:
    hist.loc[m] = {"cor": base.loc[m, "cor_oficial"], "adicional": vigente[base.loc[m, "cor_oficial"]]}
hist = hist.sort_index()
vigente, hist.tail(3)

divergências com a base: 0 de 45


({'Verde': 0.0, 'Amarela': 18.85, 'Vermelha 1': 44.63, 'Vermelha 2': 78.77},
              cor  adicional
 mes                        
 2026-08  Amarela      18.85
 2026-09  Amarela      18.85
 2026-10    Verde       0.00)

## Projeções, probabilidade e adicional da cor esperada

In [3]:
projecoes = []
for r in cen.itertuples():
    prob = backtest.probabilidade_por_grupo(A, dk, r.h, r.x_esp, r.gsf_esp)
    anunciada = hist.cor.get(r.alvo)
    projecoes.append({
        "alvo": str(r.alvo), "h": int(r.h), "anunciada": anunciada if isinstance(anunciada, str) else None,
        "prob": {c: round(float(prob[c]), 4) for c in ORD},
        "adicional": vigente[r.cor_esp],  # adicional da bandeira esperada (não há adicional proporcional)
        **{f"{k}_{n}": (round(float(getattr(r, f"{k}_{n}")), 3) if k != "cor" else getattr(r, f"{k}_{n}"))
           for n in ("esp", "li", "ls") for k in ("gsf", "pld", "x", "cor")},
    })
pd.DataFrame(projecoes)

,alvo,h,anunciada,prob,adicional,gsf_esp,pld_esp,x_esp,cor_esp,gsf_li,pld_li,x_li,cor_li,gsf_ls,pld_ls,x_ls,cor_ls
0,2026-10,1,Verde,"{'Verde': 0.6632, 'Amarela': 0.3141, 'Vermelha...",0.0,0.81,86.379,16.412,Verde,0.79,157.204,33.013,Amarela,0.82,62.384,11.229,Verde
1,2026-11,2,NaN,"{'Verde': 0.8036, 'Amarela': 0.189, 'Vermelha ...",0.0,0.92,62.875,5.030,Verde,0.91,110.034,9.903,Verde,0.92,61.997,4.960,Verde
2,2026-12,3,NaN,"{'Verde': 1.0, 'Amarela': 0.0, 'Vermelha 1': 0...",0.0,0.98,61.168,1.223,Verde,0.96,173.465,6.939,Verde,0.98,57.310,1.146,Verde
3,2027-01,4,NaN,"{'Verde': 1.0, 'Amarela': 0.0, 'Vermelha 1': 0...",0.0,1.09,65.119,0.000,Verde,1.04,296.017,0.000,Verde,1.09,57.310,0.000,Verde
4,2027-02,5,NaN,"{'Verde': 1.0, 'Amarela': 0.0, 'Vermelha 1': 0...",0.0,1.10,76.489,0.000,Verde,1.03,326.847,0.000,Verde,1.10,57.310,0.000,Verde
5,2027-03,6,NaN,"{'Verde': 1.0, 'Amarela': 0.0, 'Vermelha 1': 0...",0.0,1.09,64.382,0.000,Verde,1.03,294.741,0.000,Verde,1.09,57.310,0.000,Verde


## Insumos: ENA, GSF e PLD realizados e projetados

In [4]:
meses_hist = [m for m in base.index if dk - 17 <= m <= dk and pd.notna(base.loc[m, "gsf_band_boletim"])]
mlt_hist = base.ena_sin / (base.ena_mlt_sin / 100)
mlt_proj = ecmwf.referencia_climatologia(dk, base)
insumos = {
    "realizado": [{"mes": str(m), "ena": round(float(base.loc[m, "ena_sin"])), "mlt": round(float(mlt_hist[m])),
                   "gsf": float(base.loc[m, "gsf_band_boletim"]), "pld": round(float(base.loc[m, "pld_gatilho_boletim"]), 2)}
                  for m in meses_hist],
    "projetado": [{"mes": str(a), "mlt": round(float(mlt_proj[a])),
                   **{f"ena_{n}": round(float(refs[n][a])) for n in refs},
                   **{f"{k}_{n}": p[f"{k}_{n}"] for n in ("esp", "li", "ls") for k in ("gsf", "pld")}}
                  for a, p in zip(cen.alvo, projecoes)],
}
pd.DataFrame(insumos["projetado"])

,mes,mlt,ena_esp,ena_li,ena_ls,gsf_esp,pld_esp,gsf_li,pld_li,gsf_ls,pld_ls
0,2026-10,42800,58492,40971,76393,0.81,86.379,0.79,157.204,0.82,62.384
1,2026-11,50441,65905,52800,79661,0.92,62.875,0.91,110.034,0.92,61.997
2,2026-12,73589,65096,51753,81303,0.98,61.168,0.96,173.465,0.98,57.310
3,2027-01,102584,80488,62404,96871,1.09,65.119,1.04,296.017,1.09,57.310
4,2027-02,116022,87397,62387,104447,1.10,76.489,1.03,326.847,1.10,57.310
5,2027-03,117089,85500,68252,104037,1.09,64.382,1.03,294.741,1.09,57.310


## Escolha das séries (por alvo e cenário) e cálculo do GSF

In [5]:
K, MV = backtest.K_PADRAO, backtest.MEIA_VIDA_PADRAO
periodos = [dk + i for i in range(7)]
ena_todas = pr["ena"][periodos]
selecao = {"periodos": [str(p) for p in periodos],
           "faixa": {q: [round(float(v)) for v in ena_todas.quantile(qq)] for q, qq in (("p10", 0.1), ("p50", 0.5), ("p90", 0.9))},
           "referencias": {n: [round(float(refs[n][p])) for p in periodos] for n in refs},
           "alvos": {}}
for alvo in cen.alvo:
    traj = [dk + i for i in range((alvo - dk).n + 1)]
    selecao["alvos"][str(alvo)] = {}
    for n, ref in refs.items():
        idx = backtest._escolher(pr, ref, alvo, K, MV)
        selecao["alvos"][str(alvo)][n] = {
            "series": [int(s) for s in idx],
            "ena": [[round(float(v)) for v in pr["ena"].loc[s, traj]] for s in idx],
            "gsf": [round(float(v), 3) for v in pr["gsf"].loc[idx, alvo]],
            "pld": [round(float(v), 1) for v in pr["pld"].loc[idx, alvo]],
        }

pasta = newave.caminho_deck(dk.year, dk.month)
gh = backtest._tabela_periodos(newave.ler_nwlistop(pasta / "ghtotsin.out"))
entradas = backtest.ler_entradas_deck(dk.year, dk.month)
f = regra.fatores_vigentes(dk.year, dk.month)
gf_band = float(base.loc[dk, "gf_band_boletim"])
def passos_gsf(alvo, cenario):
    idx = selecao["alvos"][str(alvo)][cenario]["series"]
    gh_ons, ande, pch = float(gh.loc[idx, alvo].mean()), float(entradas["ande"][alvo]), float(entradas["pch"].loc[alvo].sum())
    gh_cg = (gh_ons - ande) * (1 - f["perda_bruta_conexao"]) * (1 - f["perda_conexao_cg"])
    pch_cg = pch * f["part_pch_mre"] * (1 - f["perda_pch_cg"])
    return {"alvo": str(alvo), "gh_ons": round(gh_ons), "ande": round(ande), "perdas": round(gh_ons - ande - gh_cg),
            "pch": round(pch), "pch_mre": round(pch_cg), "gh_band": round(gh_cg + pch_cg), "gf_band": round(gf_band),
            "gsf": round((gh_cg + pch_cg) / gf_band, 3)}

calculo_gsf_cenarios = {n: [passos_gsf(a, n) for a in cen.alvo] for n in refs}
calculo_gsf = calculo_gsf_cenarios["esp"]  # usado pela página v1
for n in refs:
    assert all(abs(c["gsf"] - p[f"gsf_{n}"]) < 0.006 for c, p in zip(calculo_gsf_cenarios[n], projecoes))
fatores = {k: round(v * 100, 3) for k, v in f.items()}
pd.DataFrame(calculo_gsf)

,alvo,gh_ons,ande,perdas,pch,pch_mre,gh_band,gf_band,gsf
0,2026-10,46107,3438,1581,3137,1917,43005,52869,0.813
1,2026-11,51576,3360,1787,3287,2009,48438,52869,0.916
2,2026-12,54857,3526,1902,3659,2236,51665,52869,0.977
3,2027-01,60528,3126,2127,3929,2401,57676,52869,1.091
4,2027-02,60849,3223,2135,3992,2440,57930,52869,1.096
5,2027-03,60372,3023,2125,3864,2361,57585,52869,1.089


## Desempenho (backtest sem vazamento) e trajetória das previsões no histórico

In [6]:
A["pers"] = A.deck.map(hist.cor)
A["saz"] = [hist.cor.get(a - 12) for a in A.alvo]
acerto = {n: [round(float((g[c] == g.cor_oficial).mean()), 3) for _, g in A.groupby("h")]
          for n, c in (("modelo", "cor_media"), ("persistencia", "pers"), ("ano_anterior", "saz"))}
oraculo = pd.read_pickle(load.DATA_PROCESSED / "backtest_ecmwf.pkl")["oráculo"]
acerto["oraculo"] = [round(float((g.cor_media == g.cor_oficial).mean()), 3) for _, g in oraculo.groupby("h")]

# confiabilidade: probabilidade dada (σ por grupo) × frequência observada
pares = []
for r in A.itertuples():
    p = backtest.probabilidade_por_grupo(A, r.deck, r.h, r.x_medio, r.gsf_medio)
    if p.isna().all():
        continue
    pares += [(float(p[c]), float(c == r.cor_oficial)) for c in ORD]
pares = pd.DataFrame(pares, columns=["p", "ocorreu"])
pares["faixa"] = pd.cut(pares.p, [-0.001, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
conf = pares.groupby("faixa", observed=True).agg(prevista=("p", "mean"), observada=("ocorreu", "mean"), n=("p", "size"))
confiabilidade = [{"prevista": round(float(r.prevista), 3), "observada": round(float(r.observada), 3), "n": int(r.n)} for r in conf.itertuples()]

faixa = json.loads((load.RAIZ / "reports" / "pagina" / "dados_faixa.json").read_text())
R = [x for x in faixa if x["real"]]
NIV = {c: i for i, c in enumerate(ORD)}
cobertura = sum(NIV[x["ls"]] <= NIV[x["real"]] <= NIV[x["li"]] for x in R) / len(R)

previstas = {}
for r in A[A.h.isin([1, 3])].itertuples():
    previstas.setdefault(str(r.alvo), {})[f"h{r.h}"] = r.cor_media
desempenho = {"acerto_por_h": acerto, "n_previsoes": int(len(A)), "n_decks": int(A.deck.nunique()),
              "periodo": [str(A.deck.min()), str(A.deck.max())], "cobertura_li_ls": round(cobertura, 3),
              "confiabilidade": confiabilidade, "previstas": previstas}
pd.DataFrame(acerto, index=range(1, 7)).T, conf

(                  1      2      3      4      5      6
 modelo        0.750  0.667  0.733  0.400  0.533  0.308
 persistencia  0.562  0.333  0.267  0.200  0.133  0.154
 ano_anterior  0.500  0.467  0.333  0.200  0.200  0.154
 oraculo       0.688  0.667  0.733  0.667  0.800  0.786,
                prevista  observada   n
 faixa                                 
 (-0.001, 0.1]  0.011741   0.052083  96
 (0.1, 0.3]     0.211706   0.391304  23
 (0.3, 0.5]     0.390192   0.272727  22
 (0.5, 0.7]     0.590094   0.473684  19
 (0.7, 0.9]     0.776060   0.666667   3
 (0.9, 1.0]     0.992906   0.823529  17)

## Rodadas anteriores (evolução das projeções)

Uma linha por rodada (deck + ECMWF do mesmo mês): cor esperada, probabilidade, cenários seco e úmido. Cada rodada
publicada fica guardada em `reports/pagina/historico_projecoes.json` (a projeção como foi publicada, sem recálculo); rodadas
anteriores à primeira publicação vêm do backtest sem vazamento (`03c`, `dados_faixa.json`). Rodar de novo no mesmo mês
substitui só a rodada desse mês.

In [7]:
AB = {"Verde": "Vd", "Amarela": "Am", "Vermelha 1": "V1", "Vermelha 2": "V2"}
def br(v, casas=0):
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
arquivo = load.RAIZ / "reports" / "pagina" / "historico_projecoes.json"
publicadas = json.loads(arquivo.read_text()) if arquivo.exists() else {}
publicadas[str(dk)] = [{"deck": str(dk), "alvo": p["alvo"], "h": p["h"], "esp": p["cor_esp"], "li": p["cor_li"], "ls": p["cor_ls"],
                        "prob": " ".join(f"{AB[c]} {v:.0%}" for c, v in p["prob"].items() if v >= 0.05),
                        "gp": f"GSF {br(p['gsf_esp'], 2)} · R$ {br(p['pld_esp'])}", "publicada_em": date.today().isoformat()}
                       for p in projecoes]
arquivo.write_text(json.dumps(dict(sorted(publicadas.items())), ensure_ascii=False, indent=1))

def cor_oficial(alvo):
    c = hist.cor.get(pd.Period(alvo, "M"))
    return c if isinstance(c, str) else None

rodadas = [x for x in faixa if x["deck"] not in publicadas]
for lista in publicadas.values():
    rodadas += [{**x, "real": cor_oficial(x["alvo"]), "publicada": True} for x in lista]
pd.DataFrame(rodadas).groupby("deck").size()

deck
2025-01    6
2025-02    6
2025-03    6
2025-04    6
2025-05    6
2025-06    6
2025-07    5
2025-08    6
2025-11    6
2025-12    6
2026-01    6
2026-02    6
2026-03    6
2026-04    6
2026-05    6
2026-09    6
dtype: int64

## Exporta

In [8]:
dados = {
    "gerado_em": date.today().isoformat(), "deck": str(dk), "rodada_ecmwf": str(dk),
    "adicional_vigente": vigente, "limites_x": regra.LIMITES_X, "fatores": fatores,
    "pld_min": float(base.loc[dk, "pld_min_boletim"]), "pld_max": float(base.loc[dk, "pld_max_boletim"]),
    "projecoes": projecoes, "historico": [{"mes": str(m), "cor": r.cor, "adicional": float(r.adicional),
                   **({"gsf": float(base.loc[m, "gsf_band_boletim"]), "pld": round(float(base.loc[m, "pld_gatilho_boletim"]), 2)}
                      if m in base.index and pd.notna(base.loc[m, "gsf_band_boletim"]) else {})}
                  for m, r in hist.iterrows()],
    "insumos": insumos, "selecao": selecao, "calculo_gsf": calculo_gsf, "calculo_gsf_cenarios": calculo_gsf_cenarios, "desempenho": desempenho, "rodadas": rodadas,
}
destino = load.RAIZ / "reports" / "pagina" / "produto_dados.js"
destino.write_text("window.DADOS = " + json.dumps(dados, ensure_ascii=False) + ";\n")
destino, round(destino.stat().st_size / 1024)

(PosixPath('/Users/arthur/projects/forecast-bandeiras/reports/pagina/produto_dados.js'),
 52)